# 01 — Conversando com um LLM local (sem RAG)

**LLM** (*Large Language Model*) é um modelo treinado para prever o próximo pedaço de texto.
Ele sabe muito sobre o mundo em geral, mas **nada** sobre as suas faturas.

O **Ollama** é um servidor que roda modelos no seu computador e expõe uma API em
`http://localhost:11434`. Nada sai da sua máquina. O pacote Python `ollama` só faz
requisições HTTP para esse servidor.

> Se alguma célula der `OllamaIndisponivel`, abra o app Ollama e rode de novo.

In [ ]:
from faturas.dados import carregar_faturas
from faturas.llm import MODELO_CHAT, conversar

print("modelo:", MODELO_CHAT)
print(conversar([{"role": "user", "content": "Explique o que é RAG em 2 frases, em português."}]))

## Mensagens e papéis

Uma conversa é uma lista de mensagens, cada uma com um papel (`role`):

- `system`: instruções gerais (personalidade, regras);
- `user`: o que você pergunta;
- `assistant`: o que o modelo respondeu antes (é assim que ele "lembra" da conversa).

O modelo não guarda estado entre chamadas: **toda** chamada reenvia a conversa inteira.

In [ ]:
mensagens = [
    {"role": "system", "content": "Você é um assistente financeiro direto. Responda em no máximo 2 frases."},
    {"role": "user", "content": "Vale a pena parcelar compras pequenas no cartão?"},
]
print(conversar(mensagens))

## Por baixo dos panos: streaming

`conversar` espera a resposta completa. Com `stream=True` o servidor manda a resposta
em pedaços (tokens) assim que são gerados — é o efeito "digitando" dos chats.

In [ ]:
import ollama

for pedaco in ollama.chat(
    model=MODELO_CHAT,
    messages=[{"role": "user", "content": "Liste 3 dicas curtas para economizar no mercado."}],
    stream=True,
    think=False,
):
    print(pedaco.message.content, end="", flush=True)

## Experimento 1: perguntar sobre as faturas **sem dar os dados**

O modelo nunca viu suas faturas. Observe o que ele faz: inventa um valor
(**alucinação**) ou admite que não sabe?

In [ ]:
pergunta = "Quanto gastei no Pizzaria Exemplo em outubro de 2026?"
print(conversar([{"role": "user", "content": pergunta}]))

Repare também que o modelo pode achar que está em outro ano: ele só conhece o mundo até a
data do treino (*knowledge cutoff*). Ele não tem relógio — se a data importa, ela precisa estar no prompt.

## Experimento 2: colar os dados no prompt

A forma mais simples de dar conhecimento ao modelo é **colocar a informação no prompt**.
Vamos colar as 20 transações mais recentes como texto.

In [ ]:
df = carregar_faturas("../data")


def como_texto(linha):
    return f"{linha['date']:%Y-%m-%d} | {linha['title']} | R$ {linha['amount']:.2f} | {linha['tipo']}"


recentes = "\n".join(como_texto(l) for _, l in df.tail(20).iterrows())
print(recentes)

In [ ]:
prompt = f"Estas são minhas transações do cartão:\n\n{recentes}\n\nPergunta: {pergunta}"
print(conversar([{"role": "user", "content": prompt}]))

Agora ele acerta — porque a resposta estava **no texto que demos**.

## Por que não colar tudo?

O modelo só enxerga uma quantidade limitada de texto por vez: a **janela de contexto**,
medida em *tokens* (pedaços de palavra). No Ollama, o padrão aqui é **4096 tokens**.
Uma aproximação comum é 1 token ≈ 4 caracteres.

In [ ]:
tudo = "\n".join(como_texto(l) for _, l in df.iterrows())
tokens_estimados = len(tudo) / 4
print(f"{len(df)} transações ≈ {tokens_estimados:,.0f} tokens (janela: 4.096)".replace(",", "."))

Não cabe — e mesmo com uma janela maior, prompts gigantes ficam lentos e o modelo
se perde no meio do texto.

**A ideia do RAG** (*Retrieval-Augmented Generation*): em vez de colar tudo, **buscar**
só as transações relevantes para a pergunta e colar apenas essas. É o que fazemos no notebook 03.